# 세부 과제 1 - ResNet로 새로운 이미지 데이터셋 분류하기

## 요약 (Summary)

- 본 회고록은 크게 두 가지 트러블 슈팅을 순차적으로 해결하는 과정으로 구성됩니다.

- 첫 번째 문제는 훈련 속도가 극도로 느려 실험 반복이 불가능한 부분을 해결하는 과정입니다. 초기 1 Epoch당 **10분 이상** 소요되던 문제를 1 Epoch당 **16~40초** 수준까지 단축했습니다.
    - 주요 해결 방안은 다음과 같습니다.
        - Runtime 변경
        - PyTorch에 device 명시적으로 설정
        - Mixed Precision 적용
        - 입력 이미지 크기 축소 등
    - 이 과정에서 아래의 실질적인 인사이트를 얻을 수 있었습니다.
        - 단순히 GPU를 켜는 것과 PyTorch에 실제로 적용하는 것이 별개의 계층이라는 점
        - Colab GPU Runtime 변경 시 CPU 코어 수도 함께 증가한다는 점

- 두 번째 문제는 모델이 학습은 되지만 일반화 성능이 극도로 낮은 상태를 해결하는 과정입니다. 초기 **7% -> 10% -> 약 60%**이던 문제를 **81.43%**의 Test Accuracy를 달성했습니다.
    - 주요 해결 방안은 다음과 같습니다.
        - Softmax 중복 사용 버그 수정
        - 데이터셋을 실제 이미지(CIFAR10)로 변경 (변경한 후에도 여전히 60%)
        - ResNet50 학습에서 ResNet18로 경량화
        - Scratch에서 Pretrained Weight를 적용

- 전체 과정은 체계적인 문제 정의 &rarr; 계층별 원인 분석 &rarr; 우선순위 기반 해결이라는 접근 방식으로 진행
- 단순한 코드 수정이 아닌 인프라, 프레임워크, 코드 수준의 다층적 이해가 필요했음을 보여주는사래


## 트러블 슈팅 1

- (현상) 훈련 시, 1 Epoch를 출력하기까지 10분을 초과함
    - 총 10 Epochs 예정이므로, 1시간 소요 예상

### 인간이 오래 걸린다고 느끼는 기준

|인간이 느끼는 상태|1 Epoch 기준 시간 (현재 워크로드)|주요 원인 및 심리 상태|개발 생산성 영향|
|-|-|-|-|
|즉각적 / 좋음|~30초 이내|출력이 빠르게 나오고, 기다리는 동안에도 생각을 이어갈 수 있음|최고의 몰입 상태 유지|
|안정적 (Stable)|30초 ~ 2분|커피 한 잔, Slack 확인, 간단한 메모 정도는 가능. "기다릴 만하다"는 느낌|실험 반복에 무리가 없음|
|조금 불편함|2분 ~ 5분|기다리다 보면 다른 탭을 열게 됨. "조금 느리네" 하는 수준|생산성 저하 시작|
|성가심 (Annoying)|5분 ~ 12분|기다리는 동안 집중이 완전히 깨짐. 다른 작업을 시작하게 됨|실험 주기가 길어져 동기 저하|
|고통스러움 (Painful)|12분 이상|"이걸 언제 끝나지?" 하는 생각이 지배적. 실험 자체를 꺼리게 됨|실험 횟수가 급격히 줄어듦|

- 대부분 피드백 지연이 2분을 넘기면 느리다고 느낌.
- 단순히 10 Epoch일 경우, 훈련 시간이 1시간 이상이 소요
- 시간 단축하는 여러 방법 고민 필요

### 현실적인 훈련 시간 기준 수립

1. (Step 1) 현재 워크로드 규모 요약
    - 학습 데이터: 1000장
    - batch_size=32: 약 31~32 batches per epoch
    - 모델: ResNet50
    - (소결) 전체적으로 가벼운 학습 workload (ImageNet 전체를 학습하는 수준이 아님)

2. (Step 2) 현실적인 훈련 시간 기준 수립 원칙
    - 주관적인 판단이 아니라, **하드웨어 대비 기대 성능**을 기준으로 수립 필요
    - 환경별로 명확한 threshold를 정의 필요

3. (Step 3) 실제 벤치마크 기반 추정
    - (가정) Colab T4 + PyTorch 기준으로 ResNet50 (batch 32, 224x224) 학습 시,
    - GPU에서 1 batch 처리 시간: 대략 150~400ms 수준
    - 32 batchs일 경우, 순수 연산 시간: 8~15초
    - DataLoader, tqdm, 기타 오버헤드 포함 시 실제 1 epoch 소요 시간은 25~70초 정도가 일반적임

4. (Step 4) 안정적인 영역 기준 정의
    - 해당 워크로드(ResNet50 + 1000장 + batch 32)에서 시간 기준
        |환경|1 Epoch 소요 시간|평가|판단 기준 및 행동|
        |-|-|-|-|
        |GPU (T4)|< 45초|매우 좋음 (Optimal)|이상적인 상태|
        |GPU (T4)|45초 ~ 1분 30초|안정적 (Stable)|이 구간이 목표|
        |GPU (T4)|1분 30초 ~ 3분|조금 느림|DataLoader / 설정 점검 필요|
        |GPU (T4)|3분 이상|비정상|"device 설정, precision, 병목 의심"|
        |CPU|< 10분|빠른 편|고사양 CPU|
        |CPU|10~20분|보통|일반적인 CPU 학습 속도|
        |CPU|20분 이상|느림|GPU 사용 강력 권장|
        - CPU 기준
            - 10분을 넘기면 GPU로 전환하는 것을 진지하게 고려해야 하는 수준
            - 이미 10 Epochs가 1시간 이상 소요할 것으로 예상되므로, GPU에서는 나올 수 없는 시간임.
        - GPU 기준
            - 1분 30초를 넘기면 "조금 느린" 영역에 들어섰다고 볼 수 있음.
            - 3분을 넘기면 명백히 비정상(device 설정, mixed precision 미사용, DataLoader 병목 등)
- (기준 수립) 결과적으로 **1분 내외**가 해당 워크로드에서 안정적이고 건강한 영역이므로, 이를 목표로 설정

### 해결 방안 정리

|개념 (Entity)|계층|설명|
|-|-|-|
|Infrastructure Fix|기반 환경|Colab Runtime + GPU 할당 + Device 바인딩|
|Code-level Optimization|구현 수준|device 이동, loss 함수 수정, progress bar, mixed precision"|
|Feedback Improvement|사용자 경험|tqdm, per-batch logging 등 즉각적 피드백 제공|
|Alternative Strategy|우회/대안|모델 경량화, 데이터 축소, 실험 전략 변경|
|Workflow Redesign|개발 프로세스|실험 주기를 짧게 유지하는 전체적인 접근 방식|
- 문제 해결은 단일 원인이 아니라 여러 계층의 결합으로 이루어짐
- 가장 효과적인 해결책은 **Infrastructure Fix + Code-level Optimization + Feedback Improvement 동시 적용**

1. Tier 1: 즉각 해결책 (반드시 필요)
    |순위|방안|예상 효과|난이도|비고|
    |-|-|-|-|-|
    |1|Colab Runtime을 GPU로 변경|매우 높음|낮음|가장 근본적인 해결|
    |2|코드에 device 설정 추가 (model.to(device), inputs.to(device))|매우 높음|낮음|GPU를 써도 이게 없으면 의미 없음|
    |3|Softmax 제거 (CrossEntropyLoss와 함께 사용 시 버그)|중간|낮음|학습 정확도에도 영향|
    |4|tqdm 추가 (진행률 + 실시간 loss 표시)|체감 효과 매우 높음|낮음|인간이 느끼는 "느림"을 크게 줄여줌|

2. Tier 2: 추가 성능 최적화 (Tier 1 적용 후 고려)
3. Tier 3: 대안적, 우회적 접근 (필요 시)
4. Tier 4: 개발 워크플로우 관점 해결책 (재발 방지)

- 즉각 해결책 순위 1번부터 순차적으로 진행

### 해결 방안 1 - Colab Runtime 변경

1. 사용 가능한 Colab 목록
    - Colab Pro+에서 사용 가능한 하드웨어 가속기
        - CPU
        - GPU: H100, A100, L4, T4, G4
        - TPU: v5e-1, v6e-1

2. 현재 워크로드 특성
    - 모델: ResNet50 (합성곱 연산이 지배적)
    - 데이터셋: 1000장
    - 데이터: 이미지(244 x 244)
    - 연산 특성: 행렬 곱셈 + 합성곱이 많음

3. 선택 이유
    |하드웨어|유형|CNN 학습 적합도|이 워크로드에서의 체감 속도|설정 난이도|추천도 (이번 경우)|비고|
    |-|-|-|-|-|-|-|
    |H100|GPU|최고|가장 빠름|쉬움|★★☆☆☆|과잉 + 크레딧 소모 큼|
    |A100|GPU|매우 높음|매우 빠름|쉬움|★★★☆☆|고성능이지만 과잉|
    |L4|GPU|매우 높음|매우 빠름|쉬움|★★★★☆|T4보다 신형, 성능 좋음|
    |T4 ✅|GPU|높음|빠름|쉬움|★★★★★ (강력 추천)|가장 안정적이고 무난|
    |G4|GPU|중간|보통|쉬움|★★☆☆☆|T4보다 약간 낮은 성능|
    |v5e-1 / v6e-1|TPU|중간~높음|빠름 (설정 후)|어려움|비추천|PyTorch + XLA 설정 필요|
    |CPU|CPU|매우 낮음|매우 느림|매우 쉬움|비추천|현재 상태|
    - GPU가 행렬 연산 처리 성능이 좋고, 설정 난이도가 낮음.
    - TPU는 행렬 연산 자체는 빠르지만, CNN 구조와의 호환성/최적화가 GPU만큼 좋지 않음.
    - CPU는 행렬 연산 처리 성능이 나쁨.
    - **T4 GPU** 선택
        - 과잉하지 않고, 안정적이고 무난
        - 현재 작은 워크로드이므로 T4로도 무난
        - 1 epoch 기준 30~70초 수준 예상

4. 결과 및 한계
    - (결과) 1 Epoch당 10분 &rarr; 1 Epoch당 1분 30초 감소 (약 6.67배 감소)
    - (한계) 목표는 훈련 전체가 1분 내외가 되는 게 목표이므로 추가적인 성능 개선 필요

5. 결과 근거 (CPU를 켜는 것 vs GPU를 켜는 것, Code/System Level)
    - Colab VM(가상머신)의 하드웨어 구성 단계에서 차이가 결정
        |항목|(Before) CPU Runtime|(After) GPU Runtime (T4 등)|
        |-|-|-|
        |VM 생성 시점|Google Control Plane이 CPU-only VM을 생성|Google Control Plane이 GPU가 attach된 VM을 생성|
        |하드웨어 연결|GPU가 물리적으로 연결되지 않음|NVIDIA GPU가 PCIe를 통해 VM에 연결됨|
        |OS 수준에서 보이는 장치|/dev/nvidia* 장치 파일이 존재하지 않음|/dev/nvidia0, /dev/nvidiactl, /dev/nvidia-uvm 등이 생성됨|
        |드라이버 / CUDA Runtime|NVIDIA 드라이버와 CUDA 라이브러리가 로드되지 않음|NVIDIA 드라이버 + CUDA runtime이 로드됨|
        |Python 프로세스 관점|torch.cuda.is_available() → False|torch.cuda.is_available() → True|
        |코드 실행 위치 결정 주체|OS + 하드웨어 구성|OS + 하드웨어 구성|
        - 요약
            - GPU를 켜는 것: Google이 VM에 물리적 GPU를 연결해주는 인프라 구성 작업
            - 이 단계가 끝나면 OS는 GPU를 인식하고, CUDA API 호출이 가능
        - 하지만 이 단계만으로는 PyTorch가 실제로 GPU를 사용하지 않음.

### 해결 방안 2 - 코드 내 device 설정 추가

- 코드가 GPU를 제대로 활용하지 못하거나, 다른 병목이 존재하는 상태
- 모델과 데이터를 명시적으로 GPU로 이동시킬 필요

1. 코드 내 device 설정 추가
    1. `device` 변수 설정
    2. 모델을 GPU로 이동: `.to(device)`
    3. 데이터(입력, 라벨)를 GPU로 이동: `.to(device)`

2. 결과
    - (결과) 1 Epoch당 1분 30초 &rarr; 1 Epoch당 9.78초로 감소 (약 9.2배 감소)

3. 결과 근거 (GPU를 켜는 것 vs GPU를 사용하는 것, Framework Level)
    - GPU가 OS에 보이더라도, PyTorch는 기본적으로 CPU에서 연산을 수행
    - PyTorch는 device placement라는 개념을 갖고 있음
        - 모든 torch.Tensor와 nn.Module은 어떤 device에서 관리되고 연산될지를 명시적으로 갖고 있음
        - 기본값은 cpu
        - 사용자가 명시적으로 .to("cuda") 또는 .cuda()를 호출하기 전까지는 GPU가 OS에 붙어 있어도 PyTorch는 CPU device를 사용
    - 구체적인 동작 차이
        |단계|(Before) GPU가 OS에 붙어 있는 상태 (런타임 GPU)|(After) .to(device)를 적용한 상태|
        |-|-|-|
        |Tensor 생성 위치|기본적으로 CPU 메모리 (host memory)|GPU 메모리 (device memory)|
        |연산 실행 장치|CPU에서 실행 (기본)|CUDA kernel이 GPU에서 실행|
        |데이터 이동|필요할 때마다 CPU ↔ GPU 복사 발생 (매우 비효율적)|이미 GPU에 있으므로 복사 최소화|
        |Kernel Launch|CPU에서 CUDA kernel 호출|GPU에서 직접 실행|
        |성능|GPU를 거의 사용하지 못함|GPU의 연산 능력을 제대로 사용|
        - Colab(또는 클라우드)에서 GPU를 켜는 것은 인프라 계층의 일
        - 반면 PyTorch에서 GPU를 사용하는 것은 프레임워크 실행 계층(Framework Execution Layer)의 일
        - 이 둘은 완전히 독립적인 계층이며, 아래와 같은 구조로 나뉨
            ```bash
                [Colab Control Plane]
                        ↓ (VM 생성 + GPU attach)
                [Guest OS + NVIDIA Driver + CUDA Runtime]
                        ↓ (GPU가 /dev/nvidia* 로 보임)
                [PyTorch (C++ backend + CUDA extension)]
                        ↓ (기본 device = CPU)
                [사용자 코드]
                        ↓ (.to("cuda") 호출 여부)
                실제 실행 장치 결정
            ```

### CPU 상태 비교 - Colab CPU일 때 vs Colab T4 GPU일 때

|항목|(Before) CPU Runtime|(After) GPU Runtime|차이|
|-|-|-|-|
|CPU 수|2개|12개|6배 차이|
|On-line CPU(s)|0, 1|0-11|-|
|Model name|Intel Xeon @ 2.20GHz|Intel Xeon @ 2.20GHz|동일|
|Core(s) per socket|1|6|-|
|Socket(s)|2|2|동일|
|GPU|없음|NVIDIA L4|-|

- GPU Runtime으로 변경했을 때 CPU 코어 수가 크게 증가한 것 확인 가능 (2개 -> 12개)
- 이런 차이가 발생하는 이유
    - Colab의 리소스 할당 정책
        - Google Colab은 런타임 유형에 따라 VM 스펙을 다르게 배정
        - 특히 GPU Runtime을 선택할 경우, 아래와 같은 이유로 더 많은 vCPU를 할당함
            - 딥러닝 학습에서 GPU가 연산하는 동안, CPU는 데이터를 준비해야 함(DataLoader, 이미지 전처리, augmentation 등).
            - CPU가 약하면 GPU가 기다리게 되고, 결국 GPU 활용률이 떨어짐.
            - 효과적으로 GPU를 사용하기 위해 CPU도 강력해야 할 필요.

### 트러블 이슈 1 최종 정리

1. Epoch 당 10분 초과에서 1 Epoch 당 1분 30초로 감소
   - 근거: CPU Runtime에서 GPU Runtime으로 변경했기 때문
   - 상세 설명:
        1. Runtime은 기본적으로 코드가 실행되는 가상머신 + 소프트웨어 환경 전체
        2. Runtime을 변경하면 기존 VM을 폐기하고 새로운 VM 전체를 새로 할당
        3. 변경된 새로운 VM은 GPU가 추가될 뿐 아니라, 더 좋은 성능의 CPU도 함께 할당됨
        4. 성능이 개선된 이유는 사실, GPU 때문이 아니라, GPU Runtime으로 변경하면서 더 강력한 CPU가 할당되었기 때문(아직 PyTorch가 GPU 직접 사용 전단계)

2. Epoch 당 1분 30초에서 1 Epoch 당 9.78초로 감소
    - 근거: GPU를 Framework Level에 적용했기 때문
    - 상세 설명:
        1. GPU Rumtime으로, OS에 GPU가 있더라도 PyTorch에 추가 설정을 해주지 않으면 여전히 CPU로 연산 수행
        2. PyTorch의 device placement가 기본값으로 cpu이기 때문
        3. 명시적으로 개발자가 GPU를 적용하면서, 연산 성능이 향상되었기 때문

## 트러블 슈팅 2

### 주요 문제 1 - 테스트 결과가 만족스럽지 않을 때 어떻게 대응해야 하는가? (Test Accuracy가 낮을 경우, 원인을 체계적으로 디버깅하는 방법)

- Test Accruacy 현황 분석
    - 결과: 현재 정확도: 7%
    - 분석: num_classes는 10개 클래스이므로, 10개 기준으로 거의 학습이 안 된 상태로 확인

- 디버깅 체크리스트
    1. Step 1 - 가장 의심스러운 원인부터 확인 (최우선)
    2. Step 2 - 학습이 실제로 진행되고 있는지 확인
    3. Step 3 - 데이터 문제 확인
    4. Step 4 - 평가 코드 자체에 문제가 없는지 확인
    5. Step 5 - 추가 진단 방법 탐색

1. Step 1 - 가장 의심스러운 원인부터 확인 (최우선)
    - 현황
        - 10개 클래스에서 7%의 정확도는 모델이 거의 학습하지 못한 상태를 의미 (`정확도 <= 1/클래스 수`)
        - 이 증상은 Softmax를 CrossEntropyLoss 앞에 잘못 사용했을 떄 매우 전형적으로 나타남
    - 원인 추적
        - 이 버그가 발생했을 때 나타나는 증상이 정확히 일치하기 때문
            - (훈련 과정) CrossEntropyLoss는 raw logits를 입력으로 받도록 설계되어 있음
            - CorssEntropyLoss는 내부적으로 LogSoftmax를 포함하고 있기 때문
            - 그런데 이미 Softmax를 통과한 확률값을 넣으면, loss 함수 내부 계산이 완전히 틀어짐.
            - 결과적으로 gradient가 거의 의미 없이 흐르고, 모델이 제대로 학습되지 않음.
            - 이 조합은 학습이 거의 안 되는 현상을 가장 강력하게 유발하는 버그 중 하나
        - 발생 빈도가 매우 높은 실수이기 때문
            - 실제 딥러닝을 처음하거나 중급자 수준에서도 이 실수를 하는 사람이 상당히 많음.
            - 특히 아래와 같은 생각에서 비롯됨
                - "모델 마지막에 Softmax를 넣어야 classification이 된다"
                - TensorFlow/Keras에서 쓰던 습관(Dense + softmax activation)을 그대로 PyTorch에 적용
                - "activation function을 넣어야지"라는 막연한 인식
    - 해결 방향
        - 모델의 forward 메서드에서 Softmax 레이어를 제거하고
        - forward에서는 logits만 반환하게 수정하기
        - `nn.Sequential` 안에 Softmax를 넣지 않는 것 권장

2. Step 2 - 학습이 실제로 진행되고 있는지 확인
    - 의문
        - 모델이 실제로 학습을 하고 있는지, loss가 제대로 떨어지고 있는지 확인 필요
    - 추천 확인 방법
        - 매 epoch마다 Average Loss를 출력
        - 가능하면 tqdm을 사용하여 실시간 loss 변화를 관찰
    - 판단 기준
        - Loss가 점점 감소해야 정상
        - Loss가 거의 변하지 않거나, 오히려 증가한다면 학습이 제대로 안 되고 있다는 의미
        - 특히 Softmax + CrossEntropyLoss 조합이면 loss가 이상하게 나올 가능성

3. Step 3 - 데이터 문제 확인
    - 현황
        - 렌덤 데이터 사용
    - 문제 추적
        - 렌덤으로 만든 데이터는 실제 이미지 패턴이 거의 없음
        - 이 상태에서 모델이 의미 있는 특징을 학습하는 것은 매우 어려움
    - 확인 및 개선 방향
        - 현재는 모델 버그를 디버깅하는 용도로 적절하지만, Accuracy를 평가하기에는 부적합
        - 진짜 이미지 데이터셋(torchvision.datasets.CIFAR10 또는 ImageFolder)을 사용해서 테스트해보는 것을 권장

4. Step 4 - 평가 코드 자체에 문제가 없는지 확인
    - 확인 포인트
        - model.eval()을 호출했는가? (드롭아웃, 배치정규화 영향)
        - with torch.no_grad() 블록 안에 있는가?
        - torch.max(outputs, 1)에서 outputs가 logits인지 확인

5. Step 5 - 추가로 확인할 수 있는 진단 방법
    |진단 항목|확인 방법|기대 결과 (정상)|현재 의심 수준|
    |-|-|-|-|
    |학습 Loss 감소 여부|매 epoch loss 출력|점진적 감소|높음|
    |Softmax 제거 여부|모델 forward 마지막 확인|Softmax 없어야 함|매우 높음|
    |데이터가 실제 이미지인가|데이터 생성 코드 확인|실제 이미지 데이터 권장|높음|
    |학습률이 적절한가|lr=0.0001 정도 사용 중|보통 적절|중간|
    |Pretrained 사용 여부|weights=... 사용 중인가?|사용하는 게 일반적|-|

### 해결 방안 1 - 가장 의심스러운 원인 제거 (Softmax 중복 사용)

- 해결
    - 모델의 forward 함수에서 Softmax 레이어 제거
    - forward에서는 logits만 반환하도록 수정
- 결과 (Softmax 제거 효과)
    |항목|(Before) Softmax 제거 전|(After) Softmax 제거 후|평가|
    |-|-|-|-|
    |Epoch 1 Loss|2.30|2.32|비슷|
    |Epoch 5 Loss|2.15|1.30|크게 개선|
    |Epoch 10 Loss|1.96|0.36|극적으로 개선|
    |학습 속도|매우 느림|빠르고 안정적|대성공|
    - Loss가 제대로 떨어지기 시작함
    - (결론) Softmax를 제거한 것은 정확한 판단.
- 원인
    - CrossEntropyLoss는 내부적으로 LogSoftmax + NLLLoss를 포함하고 있음
    - 따라서 입력으로 raw logits를 받아야 정상 동작
    - 이미 Softmax를 적용한 값을 넣으면 loss 계산이 수학적으로 잘못되고, gradient 왜곡 발생
- 한계
    - Test Accuracy는 7%에서 9.5%로 개선 (여전히 낮은 상태)
        - Loss가 0.36까지 떨어진 것으로 보아, 학습은 잘 되고 있으나,
        - Test Accuracy가 9.5인 것으로 보아, 일반화(Generalization)이 잘 안 되고 있음.
    - 예상 원인: 데이터가 Random Noise

### 해결 방안 2 - 데이터 문제 확인

- 현황 분석
    - (현황) 현재 사용 중인 데이터는 numpy로 만든 random noise
    - (분석) 모델 입장에서 랜덤한 숫자 + 랜덤한 라벨일 뿐
        - 이런 데이터에서는:
            - 학습 Loss는 떨어질 수 있음(모델이 noise를 외우려고 노력)
            - 하지만 Test Accurace는 크게 오르기 어려움 (진짜 패턴이 없기 때문)
    - (소결) Train Loss는 낮은데 Test 성능이 안 따라오므로, 더 훈련을 하더라도 Overfitting 가능성
- 해결 방향
    - 현재 데이터 상태에서 개선해보기
        - 학습률 조정, epoch 늘리기, regularization 추가 등
    - 실제 이미지 데이터셋으로 변경하기
        - torchvision.datasets.CIFAR10, CIFAR100 또는 ImageFolder 등을 사용해서 다시 학습
    - (소결) 훈련 데이터 자체가 noise이므로, 실제 이미지 데이터셋으로 변경하는 방법 선택
- 구체적 해결 방안
    - 실제 이미지 데이터셋으로 변경하는 방법을 선택했으므로, 이미지 데이터셋의 종류 분석 필요
    - torchvision에서 제공하는 주요 이미지 데이터셋 비교
        |데이터셋|클래스 수|이미지 크기|난이도|용도 추천 상황|특징|
        |-|-|-|-|-|-|
        |MNIST|10|28×28|매우 쉬움|아주 기초적인 실험|흑백 숫자 이미지|
        |CIFAR10✅|10|32×32|보통|ResNet 실험, 디버깅, 빠른 실험|컬러 이미지, 가장 많이 사용|
        |CIFAR100|100|32×32|어려움|모델 성능을 더 제대로 테스트하고 싶을 때|클래스 수가 많아서 학습이 어려움|
        |ImageNet|1000|224×224|매우 어려움|대형 모델 학습 (ResNet50 이상)|용량이 매우 큼 (약 150GB)|
        |ImageFolder|사용자 정의|사용자 정의|-|자신의 로컬 이미지 폴더를 사용할 때|가장 유연함|
        - ResNet50 실험에 적합하고, 빠른 테스트가 가능하므로 CIFAR10 선택
        - 추후 모델 성능을 더 테스트하고 싶을 때 CIFAR100로 확장
        - 본인 데이터셋을 만들 때 ImageFolder로 확장
    - (추가 질문) 기존 Random 데이터가 224, 224 크기인데, ImageNet도 224, 224 크기니까 객관적인 성능 평가에 더 적합한 모델이 아닐까?
        - CIFAR10에서 Resize로 해결 가능
        - ImageNet은 아래와 같은 이유로 비추천
            |단점|설명|영향도|
            |-|-|-|
            |용량이 매우 크다|학습용 이미지 약 1.28 million 장, 압축해도 150GB 이상|다운로드 자체가 부담|
            |학습 시간이 오래 걸린다|ResNet50으로 ImageNet을 학습하면 수십 시간 ~ 며칠 소요|빠른 실험/디버깅 불가능|
            |리소스 소모가 크다|GPU 메모리, 시간, 전력 모두 많이 듦|Colab 무료/Pro 사용자에게 부담|
            |디버깅이 어렵다|문제가 생겼을 때 원인을 찾기가 매우 어려움|현재 단계에 부적합|
            - 현재 목표는 모델이 제대로 훈련되는지 디버깅하기 위함.
            - ImageNet는 과도한 선택
- 결과
    - 훈련 과정에서 1 Epoch 9.78초에서 1 Epoch가 10분으로 늘어남
    - 테스트 결과를 비교 분석하기 전에 훈련 속도가 더뎌 확인 불가

### 주요 문제 2 - 훈련 속도가 더딜 떄 어떻게 대응해야 하는가?

- 현황 분석
    - 랜덤 데이터에서 실제 이미지 데이터셋 "CIFAR10"로 변경
    - 훈련 속도가 1 Epoch 당 10분으로 늘어나서, 테스트 결과 판단이 어려움
    - 해결 과제 우선 순위를 "테스트 정확성 개선"에서 "훈련 속도 개선"으로 변경

### 해결 방안 1 - 모델 연산 최적화 (Mixed Precision 적용)

- 현황 분석
    - CIFAR10로 데이터로 변경 한 뒤, 1 Epoch당 약 10분 정도 소요
    - nivida-smi 확인 결과 GPU-Util이 99%로 매우 높게 유지
    - 이는 모델 연산 자체가 여전히 무거운 상태임을 의미

- 원인 분석
    - ResNet50 + 224×224 입력 크기 조합은 T4 GPU에서 연산량이 상당함
    - FP32(32비트)로 모든 연산을 수행하면 Tensor Core를 제대로 활용하지 못함
    - 이로 인해 모델 forward/backward 연산이 전체 학습 시간의 큰 비중을 차지하게 됨

- 해결 방안
    - Mixed Precision Training(AMP)을 적용
        - PyTorch의 torch.cuda.amp을 사용하여 FP16과 FP32를 혼합해서 연산
        - autocast()로 순전파를 FP16으로 수행하고, GradScaler를 통해 Gradient를 안전하게 관리
        - Tensor Core를 적극적으로 활용하여 연산 속도를 크게 향상

- 결과
    |항목|Mixed Precision 적용 전|Mixed Precision 적용 후|개선 효과|
    |-|-|-|-|
    |1 Epoch 소요 시간|약 10분|4분 40초|약 50% 이상 단축|
    |GPU-Util|99%|99%|비슷|
    |학습 안정성|안정|안정|양호|
    - Mixed Precision 적용 후 학습 속도가 약 2배 가량 빨라짐
    - GPU를 보다 효율적으로 활용하고 있다는 의미

- 한계 및 추가 고려사항
    - Mixed Precision 적용 후에도 여전히 4분 가까이 소요
    - 근본적인 원인(입력 크기 + 모델 크기)이 여전히 존재
    - 추가적인 속도 개선을 위해서는 아래와 같은 방법 고려 가능
        - 입력 크기 줄이기(32×32, 64×64)
        - 더 가벼운 모델 사용(ResNet18 등)
        - torch.compile 적용
        - A100 이상의 고성능 GPU 사용 (보조 수단)

### 해결 방안 2 - 입력 크기 줄이기

- 현황 분석
    - Mixed Precision을 적용한 후에도 1 Epoch당 약 4분 정도 소요
    - nvidia-smi 확인 결과 GPU가 99%로 거의 풀가동 중이었으나, 전체적인 학습 속도가 여전히 느림
    - 기존 입력 데이터(렌덤)과 객관적인 비교를 위해, 입력의 크기를 224×224로 Resize하여 사용하고 있었기 때문에 모델 연산량이 과도하게 큰 상태

- 원인 분석
    - ResNet50은 원래 ImageNet(224×224) 학습을 위해 설계된 모델
    - CIFAR10 이미지를 224×224로 Resize하여 학습하는 것은 연산량이 매우 큼
    - 특히 T4 GPU와 같은 중저사양 환경에서는 입력 크기가 모델 연산 속도에 큰 영향을 줌
    - Mixed Precision을 적용하더라도, 입력 크기가 크면 그 효과가 제한적일 수 있음

- **실무 관점에서의 고찰**
    - 실무에서는 무작정 입력 크기를 줄이는 것이 아니라, 정확도와 속도(비용)의 균형을 고려하여 결정함
    - CIFAR10은 원본 해상도가 32×32인 일상 사물 이미지 데이터셋
    - **의료·위성·산업 영상처럼 고해상도가 필수인 데이터와는 성격이 다름**
    - 따라서 CIFAR10 수준의 데이터에서는 32×32 ~ 128×128 사이에서 입력 크기를 결정하는 것이 일반적
    - 고해상도 데이터(의료, 위성 등)의 경우 원본 해상도를 유지하거나, Tiling/Patch 방식 등을 사용함

- 해결 방안
    - 입력 이미지 크기를 점진적으로 축소
    - 224×224 &rarr; 160×160 &rarr; 128×128 &rarr; 32×32 순으로 입력 크기를 줄여가며 실험
    - 입력 크기를 줄이면 Convolution 연산량이 크게 감소하여 전체 학습 속도가 빨라짐
    - 특히 CIFAR10처럼 원본 해상도가 낮은 데이터셋에서는 과도하게 큰 입력 크기를 사용할 필요 없음.
    - ✅ 128×128 선택이 현재 환경에서 괜찮은 타협점으로 보임

- 한계 및 추가 고려사항
    - 입력 크기를 32×32로 줄이면 학습 속도는 크게 빨라지지만, 세밀한 특징 학습이 어려워질 수 있음
    - Pretrained ResNet50의 장점을 최대한 활용하려면 128×128 이상을 유지하는 것이 유리
    - 최종 모델 성능이 중요해지는 단계에서는 입력 크기를 다시 상향 조정하거나, 더 강력한 모델 아키텍처를 고려해야 함
    - 현재 단계(모델 디버깅 및 실험 반복 효율화)에서는 입력 크기 축소가 효과적인 전략

### 주요 문제 3 - (복귀) 테스트 성능 해결

- 현황 분석
    - "훈련 속도 개선 문제" 해결로 "테스트 성능 개선 문제"로 복귀
    - Loss가 1 Epoch일 떄 1.92, 10 Epoch일 때 0.52로 줄어듬
    - 하지만, Test Accuracy는 여전히 10.02%로 일반화 성능이 떨어짐.
    - 과적합(Overfitting) 문제 여전히 존재

### 해결 방안 1 - 소스 코드 내 대표적인 병목 지점 수정

- 현황 분석
    - Scratch란, weights를 None으로 설정하여 처음부터 weights를 학습하는 방법
    - 현재 ResNet50의 Pretrained Weight를 사용하지 않음.(전이 학습)
    - Overfitting이 발생하기 쉬운 조합
        - 모델: ResNet50 (파라미터 수 2,500만 개로 매우 큰 모델)
        - 학습 방식: weights=None (Scratch 학습)
        - 데이터: CIFAR10 (50,000장, 상대적으로 데이터 양이 적음)
        - Epoch 수: 10 Epoch (ResNet50을 scratch로 학습하기에 턱없이 부족)
    - (기타) - 현재 이미 학습된 weights를 사용하고 싶지 않은 상태

- 원인 분석
    - Overfitting이 발생하기 쉬운 조합에서 10 Epoch 만으로 제대로 된 일반화 성능을 기대하기 어려움

- 해결 방법 종류
    1. Pretrained Weight를 사용하기
        - ImageNet으로 미리 학습된 가중치를 사용하면, 이미 좋은 feature를 가지고 시작
        - Overfitting이 훨씬 적고, 일반화 성능이 빠르게 올라감
    2. 모델을 더 가볍게 변경(ResNet18 등)
        - ResNet50을 scratch로 학습하기에는 지나치게 큰 모델

- 해결 방안 선택 기준
    |항목|ResNet50 (Scratch)|ResNet18 (Scratch)|평가|
    |-|-|-|-|
    |파라미터 수|약 2,560만|약 1,170만|ResNet18이 유리|
    |Overfitting 위험|높음|상대적으로 낮음|ResNet18이 유리|
    |CIFAR10 학습 난이도|높음|보통|ResNet18이 유리|
    |10 Epoch 내 수렴|매우 어려움|상대적으로 수월|ResNet18이 유리|
    |최종 성능|보통|보통~좋음|비슷하거나 ResNet18이 약간 앞설 수 있음|
    - ResNet18의 장점
        - 파라미터 수가 적어서 Ovefitting이 덜 발생
        - CIFAR10처럼 데이터가 많지 않은 경우 더 적합
        - 학습 속도도 ResNet50보다 빠름
    - ResNet18로 변경 선택 ✅

- 한계
    - ResNet18을 사용하더라도 scratch를 사용해도 여전히 문제가 해결되지 않을 가능성 높음
        - 10 Epoch로는 여전히 부족. scratch로 CIFAR10에서 좋은 성능을 내려면 100~200 Epoch 정도 학습 필요
        - Pretrained보다 성능이 낮을 가능성. Pretrained에서는 10 Epoch 만으로 Test Accuracy 80% 기대 가능

- 결과
    - ResNet18만으로 60%대 Test Accuracy 확인
    - Scratch를 100으로 설정했더니 72%로 조금 상승 (효과가 미미)

### 해결 방안 2 - Pretrained Weight 추가

- 현황 분석
    - ResNet18만을 scratch(weights=None)로 학습한 결과, 10 Epoch 기준 Test Accuracy 60%대 머무름
    - Loss는 꾸준히 감소했으나, 일반화 성능이 목표치 만큼 개선되지 못함
    - 목표 성능인 80% 이상 달성이 어려움
    - Scratch 학습의 한계 명확하게 확인
- 원인 분석
    - 파라미터 수 대비 데이터 부족: ResNet18은 약 1,170만 개의 파라미터를 가지고 있지만, CIFAR10은 학습용 이미지가 5만 장에 불과
    - Feature 학습 부담: ImageNet처럼 대규모 데이터로 미리 학습된 feature가 없기 때문에, 모델이 low-level feature부터 처음부터 학습해야 함
    - 수렴 속도 저하: Scratch 학습은 일반적으로 수렴 속도가 느리고, 특히 10 Epoch 정도로는 충분한 일반화 성능을 기대하기 어려움
    - Overfitting 위험: 파라미터가 많은 모델을 적은 데이터로 학습할 때 Overfitting이 쉽게 발생
- 해결 방안
    - Scratch에서 Pretrained Weight 추가
    - ImageNet으로 미리 학습된 가중치를 불러와서 학습을 시작하는 방식으로 전환
    - ImageNet에서 이미 학습된 강력한 feature extractor를 활용
- 결과
    |항목|Pretrained 미적용 (Scratch)|Pretrained 적용|개선 효과|
    |-|-|-|-|
    |Test Accuracy|60%대|81.43%|+20%p 이상|
    |수렴 속도|느림|빠름|크게 개선|
    |Overfitting 위험|높음|낮음|개선|
    |10 Epoch 내 성능|제한적|우수|크게 개선|
    - Pretrained Weight를 적용한 결과, 10 Epoch 만으로 81.43%의 Test Accuracy를 달성
    - 특히 일반화 성능이 크게 개선되면서, Loss 감소와 Accuracy 상승이 동시에 이루어짐
- 한계 및 추가 고려사항
    - Pretrained를 적용했음에도 여전히 80% 중반대에 머무르고 있어, 추가 개선 여지가 존재함
    - 현재 사용 중인 커스텀 Head(fc1 → 256 → fc2)가 Pretrained feature와 잘 맞지 않을 수 있음
    - Learning Rate가 0.0001로 다소 낮게 설정되어 있어, Pretrained fine-tuning에 최적화되지 않았을 가능성
    - 추가 개선을 위해 고려할 수 있는 방향:
        - Learning Rate 상향 조정 (예: 0.001 ~ 0.01)
        - Optimizer를 SGD + Momentum으로 변경
        - 마지막 분류기(head) 구조 단순화 (model.fc 직접 교체 방식)
        - Data Augmentation 추가 (RandomCrop, RandomHorizontalFlip 등)

## 최종 회고록

- 이번 트러블 슈팅 과정을 통해 단순힌 모델을 학습시키는 행위 뿐 아니라, 문제를 체계적으로 정의하고 계층적으로 분석하며 해결하는 과정이 얼마나 중요한지 깊이 깨달았다.

- 가장 인상 깊던 점은 문제의 원인이 단일 계층에 있지 않다는 점이다. 처음에는 단순히 훈령이 느리다는 현상에 집중했지만, 실제로는 Colab의 인프라 계층(VM 할당), PyTorch의 프레임워크 계층(device placement), 그리고 코드 구현 계층이 복합적으로 작용
- 특히 GPU를 켜는 것과 PyTorch에 GPU를 사용하는 것이 완전히 다른 계층 문제라는 점을 명확하게 이해하게 된 것이 큰 수확

- 두 번째는 중요하게 배운 점은 모델 용량과 데이터셋 규모, 학습 방식 간의 균형이다. ResNet50을 Scratch로 CIFAR10에 적용했을 때 발생한 극심한 Overfitting과 낮은 일반화 성능은 단순히 모델을 크게 만드는 것이 항상 좋은 선택이 아니라는 점을 보여주었다.
    - 결국 ResNet18로 모델을 경량화하고, Pretrained Weight를 적용하면서 비로소 의미 있는 성능을 달성할 수 있었다. 이는 실무에서도 "문제 해결에 적합한 모델"을 선택하는 것이 얼마나 중요한지 상기시켜 주었다.
    - 또한 Pretrained Weight의 가치를 체감했다. Scratch 학습에서는 10 Epoch 동안 60%대에 머물렀던 성능이 Pretrained를 적용한 순간, 81%까지 상승한 것은 수치 이상의 의미를 가진다. 특히 데이터가 상대적으로 적은 상황에서 처음부터 모델을 학습시키는 것의 비효율성을 경험할 수 있었다.

- 마지막으로 문제 해결과정에서 체계적인 디버깅 체크리스트와 우선순위 설정을 중요하게 염두했다. Softmax 버그처럼 전형적이지만 놓치기 쉬운 실수를 가장 먼저 확인하고, 순서대로 데이터 문제, 모델 구조 문제로 접근한 방식이 문제를 구조적으로 해결하고 개선하는 데에 큰 도움이 되었다.

- 이번 과제를 통해 단순히 코드를 작성하고 실행하는 것이 아닌, 문제의 본질을 계층적으로 파악하고, 적절한 수준에서 해결책을 찾는 태도를 발전시키는 것이 도움된다는 것을 알 수 있었다. 또한, 모델을 선택할 때도 반드시 성능뿐만 아니라 "학습 효율성과 일반화 가능성"까지 종합적으로 고려하는 시각을 갖출 수 있었다.